# Noise Reduction Article:
https://arxiv.org/html/2502.03462v1

# Overview
QuTiP has animation functions to visualize the time evolution of quantum dynamics.

In [ ]:
# Importing all of the moduls that will be needed in future calculation
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
#import imageio_ffmpeg
import matplotlib as mpl
from qutip import ( Bloch, sigmam, sigmax, sigmay, mesolve, ket, basis, tensor, sigmaz, qeye, anim_schmidt,
                   complex_array_to_rgb, spin_q_function,
                   anim_spin_distribution, about)
from qutip.ipynbtools import plot_animation
%matplotlib inline

# Time evolution of a qubit
Consider a system composed of two qubits. Its hamiltonian is  σz⊗1 and the initial state is an entangled state (∣∣10⟩+ ∣∣01⟩)/ 1/2. This operator acts on the first qubit and leaves the second qubit unaffected.

In [ ]:
H = tensor(sigmaz(), qeye(2))

#Inital state
psi0 = (ket('10') + ket ('01')).unit()

tlist = np.linspace(0, 3 *np.pi, 100)

results = mesolve(H, psi0, tlist, [], e_ops = [])

fig, ani = anim_schmidt(results)

HTML(ani.to_jshtml())

# Titled Animation

In [ ]:
compl_circ = np.array([[(x + 1j*y) if x**2 + y**2 <= 1 else 0j
                        for x in np.arange(-1, 1, 0.005)]
                       for y in np.arange(-1, 1, 0.005)])

fig = plt.figure(figsize=(7, 3))
ax0 = plt.subplot(1, 2, 1)
ax1 = plt.subplot(1, 2, 2)
ax1.set_xlabel("x", fontsize=14)
ax1.set_ylabel("y", fontsize=14)
ax1.imshow(complex_array_to_rgb(compl_circ, rmax=1, theme='light'),
           extent=(-1, 1, -1, 1))
plt.tight_layout()
fig, ani = anim_schmidt(results, fig=fig, ax=ax0)
# add title
ax0.set_title('schmidt')
ax1.set_title('color circle')

This Cell animates the evolution of a quantum spin-1/2 particle (a qubit) as it rotates all the way around the Bloch sphere and back to its starting position.

the equation that is responsible for the states rotation is:
$$ \lvert \psi(i) \rangle = \cos\left(\frac{\pi}{2} \cdot \frac{i}{60}\right)\lvert 0 \rangle + \sin\left(\frac{\pi}{2} \cdot \frac{i}{60}\right)\lvert 1 \rangle $$

$ basis(N, m) $, where:

N is the dimension of Hilbert space

m is the index of the specified state


In [ ]:
theta = np.linspace(0, np.pi, 90)
phi = np.linspace(0, 2*np.pi, 90)

Ps = list()

for i in range(0, 121, 2):
    spin = np.cos(np.pi/2*i/60)*basis(2, 0) + np.sin(np.pi/2*i/60)*basis(2, 1)
    Q, THETA, PHI = spin_q_function(spin, theta, phi) #Husimi Q function, Q = 90 x 90 matrix elements
    Ps.append(Q)

fig, ani = anim_spin_distribution(Ps, THETA, PHI, projection = '3d', colorbar=True)
HTML(ani.to_jshtml())

# Bloch Sphere

In [ ]:
def qubit_integrate(w, theta, gamma1, gamma2, psi0, tlist):
    """To simulate the time evolution of an open quantum system (a single qubit)
      interacting with a noisy thermal environment"""
    
    # 1)Introduce the Hamiltonian & the Operators
    sx = sigmax()
    sy = sigmay()
    sz = sigmaz()
    sm = sigmam()
    H = w * (sz * np.cos(theta) + sx * np.sin(theta)) # w is the qubit angular freq. 

    #2) Introduce the collapse operators
    c_op_list = []
    n_th = 0.5 #temp.
    rate = gamma1 * (n_th + 1)

    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sm)
    rate = gamma1 * n_th
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sm.dag())
    rate = gamma2 
    if rate > 0.0:
        c_op_list.append(np.sqrt(rate) * sz)

    # let the system evolve and calculate expectation values
    output = mesolve(H, psi0, tlist, c_op_list, e_ops=[sx, sy, sz], options={"store_states": True})
    return output

In [ ]:
w = 1.0 * 2 * np.pi # qubit angular frequency
theta = 0.2 * np.pi # qubit angle from sigma_z axis (toward sigma_x axis)
gamma1 = 0.5 #qubit relaxation rate
gamma2 = 0.2 #qubit dephasing rate

# give the initial state where the evolution will start from
a = 1.0
psi0 = (a * basis(2,0) + (1-a) * basis(2, 1)) /\
      (np.sqrt(a**2 + (1 - a) ** 2))

tlist = np.linspace(0, 4, 150)

result = qubit_integrate(w, theta, gamma1, gamma2, psi0, tlist)

result

# Plotting Result

In [ ]:
def plot_setup(result):
    """To set up the parameters for plotting the sphere """

    fig = plt.figure(figsize=(8,8))
    axes = fig.add_subplot(111, projection ='3d', elev=30, azim=-40)

    return fig, axes    

sphere = None

def plot_result(result, n, fig = None, axes = None):
    """To show the evolution of the system animated"""
    global sphere

    if fig is None or axes is None:
        fig, axes = plot_setup(result)

    if not sphere:
        sphere = Bloch(axes = axes)
        sphere.vector_color = ['r']

    sphere.clear()
    sphere.add_vectors([result.expect[0][n],
                        result.expect[1][n],
                        result.expect[2][n]])
    sphere.add_points(
        [
            result.expect[0][: n + 1],
            result.expect[1][: n + 1],
            result.expect[2][: n + 1],
        ], meth = 'l',
        )
    sphere.make_sphere()
    return axes.artists


plot_animation(plot_setup, plot_result, result,  'qubit_evolution', writer='ffmpeg', codec=None)

# Investigating Purity

In [ ]:
# 1. Define simulation time parameters
tlist = np.linspace(0, 50, 200)

# 2. Run the open quantum system solver
output = qubit_integrate(w=1.0, theta=np.pi/4, gamma1=0.1, gamma2=0.05, psi0=basis(2,0), tlist=tlist)

# 3. Calculate purity at every single time step
purities = []
for state in output.states:
    # output.states contains the full density matrix (rho) for each time step
    purity_val = state.purity()
    purities.append(purity_val)

# 4. Plot the Purity Decay Curve
plt.figure(figsize=(7, 4))
plt.plot(tlist, purities, label=r"Qubit Purity ($\gamma$)", color="purple", lw=2) #r"" is a safer way to pass the Latex=like texts, it means raw string 
plt.axhline(0.5, color="black", linestyle="--", label="Maximally Mixed State (0.5)")

plt.title("Quantum Purity Decay Over Time", fontsize=12)
plt.xlabel("Time ($t$)", fontsize=10)
plt.ylabel(r"Purity = $\mathrm{Tr}(\rho^2)$", fontsize=10)
plt.ylim(0.45, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right")
plt.show()